# v106_gpu_stage1 — stage 1 retrained on the GPU with 3× the entities; two-stage on top

| Field | Value |
|---|---|
| **Version** | `v106_gpu_stage1` |
| **Plan group** | D3 / D4 (stage-1 matcher, XGBoost on the GPU), C5 (two-stage) |
| **Parent version** | v107 (v104's two-stage, rule tuned on the tight mock) |
| **Author** | M1 rajaguru2004 |
| **Date** | 2026-09-26 |
| **Status** | shortlisted |

v104's stage 1 is v101: LightGBM on 200k fit-side entities at ~60 % of the test's density.
Every train-fold entity absent from the mock fold is fair game for stage 1 (its scores on the
mock stay honest): the entities the mock drops and, for the US, those in clusters it does not
keep. v106 trains stage 1 on up to 360,000 of them (~20M pairs, what the RTX 2050's 4 GB holds)
with XGBoost on the GPU, blocked against
the train-fold pool (blocking configuration: `B6`), then rebuilds the two-stage model
of v104 on top.

```
train-fold S1 absent from the mock ─► blocking vs train-fold pool ─► features → disk chunks
    ─► XGBoost QuantileDMatrix on the GPU (streamed) ─► stage 1 v2
mock fold ─► stage 1 v2 on every pair ─► filter + competition + anchor features
    ─► stage 2 (GPU, cross-fitted on the mock's fit entities) ─► rule on mock tune ─► mock val
```

## 1. Hypothesis

* **Change vs parent (v104):** stage 1 only (and the blocking configuration, if v105 adopted
  one). More training entities (~2x–3x v101's 200k), at a denser pool (the train-fold pool), and
  a GPU learner; everything downstream (filter, competition and anchor features, stage 2,
  rule) is rebuilt on the new stage-1 probabilities exactly as in v104.
* **Why:** stage 1 sets three things v104 depends on: which candidates survive the filter,
  the competition features (ranks and rivals are stage-1 probabilities) and the anchor
  (each entity's best other candidate). A stronger stage 1 improves all three.
* **Check:** stage 1 v2 alone on the mock (threshold rule tuned on the mock tune entities,
  as v103 did for v101) against v103's 0.9704, then the two-stage against v104.
* **Stage 2 also gets** rival features (the record against its best rival S1's name and
  address; 0.94 % of true pairs are lost in 1-to-1 conflicts on v104's mock) and, when v104's
  anchor ablation pays, cohesion features; an ablation measures both.
* **Rules** are tuned on the tight mock (false merges ×1.45, calibrated on the leaderboard)
  and versions compared by `est_public`.
* **Discard if:** est_public does not beat v107 by more than 0.001.

## 2. Setup

Configurations: the pipeline (v101's feature groups; blocking as chosen), stage 1 v2 (XGBoost
on the GPU, 127 leaves, learning rate 0.1, early stopping on a 5 % id-hash slice of its own
entities) and the two-stage settings of v104.

In [ ]:
import json
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, replace

import numpy as np
import pandas as pd

from entity_resolution import config as C
from entity_resolution.blocking import TopKSpec
from entity_resolution.data import isin
from entity_resolution.decision import DecisionRule, apply_rule, decide, tune_expected
from entity_resolution.evaluate import blocking_report, error_samples
from entity_resolution.features import DEFAULT_GROUPS
from entity_resolution.mock import FP_WEIGHT, PUBLIC_OFFSET, build_mock, target_shape
from entity_resolution.model import MatcherParams
from entity_resolution.pipeline import (
    Fitted, PipelineConfig, learn_token_map, mock_scores, peak_rss_gb, tune_mock,
)
from entity_resolution.split import load_fold
from entity_resolution.stage1 import absent_from_mock, clean, fit_stage1, write_chunks
from entity_resolution.tracking import log_result, timed
from entity_resolution.trainset import inner_split, sample_s1
from entity_resolution.twostage import (
    TwoStage, TwoStageConfig, fit_stage2, mock_scored, mock_stage1, run_test_two_stage,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_columns", 30)

EXP_DIR = C.EXPERIMENTS / "v106_gpu_stage1"
ARTIFACTS = EXP_DIR / "artifacts"
cfg = PipelineConfig(feature_groups=(*DEFAULT_GROUPS, "frequency"),
                     model=MatcherParams(n_estimators=4000),
                     blocking=replace(replace(PipelineConfig().blocking, name_addr_word=replace(PipelineConfig().blocking.name_addr_word, top_k=50), max_per_s1=100, exact_max_group=200, addr_char=TopKSpec('addr_norm', 'word', (1, 2), top_k=10, min_sim=0.3, max_df=0.01, max_df_abs=10_000)), cap_order='sim_first'))
S1_PARAMS = MatcherParams(backend="xgb", device="cuda", num_leaves=127, learning_rate=0.1,
                          n_estimators=3000, early_stopping=50)
N_STAGE1 = 360000   # entities: ~20M training pairs fit the RTX 2050's 4 GB as a binned matrix
tcfg = TwoStageConfig(floor=0.01, max_cands=16, cohesion=False, rivals=True,
                      model=MatcherParams(backend="xgb", device="cuda", n_estimators=4000))
parent = json.loads((C.EXPERIMENTS / "v104_two_stage" / "metrics.json").read_text())
# v107 = v104 with the rule tuned on the tight mock: the est_public to beat
v107 = json.loads((C.EXPERIMENTS / "v107_tight_rule" / "metrics.json").read_text())
EST_TO_BEAT = v107["metrics"]["comparison"]["v107 tight rule"]["est_public"]
v103 = json.loads((C.EXPERIMENTS / "v103_mock_rule" / "metrics.json").read_text())
STAGE1_CACHE = (cfg.cache_dir / "stage1" /
                f"v106_{cfg.blocking.key()}_f{tcfg.floor}_k{tcfg.max_cands}_a{int(tcfg.anchors)}"
                f"_c{int(tcfg.cohesion)}_r{int(tcfg.rivals)}")
timings: dict[str, float] = {}
t_start = time.time()
print("blocking", cfg.blocking.key(), "| v107 est_public to beat", round(EST_TO_BEAT, 4),
      "| v104 mock F0.5", parent["mock_f05"], "| v103", v103["mock_f05"])

## 3. Data

The mock fold (as in v103–v104; its blocking under this configuration is cached or built on
first use) and the stage-1 training entities: train-fold S1 absent from the mock, capped at
`N_STAGE1` by id hash (`sample_s1`), about 20M candidate pairs.

In [ ]:
with timed("load", timings):
    train = load_fold("train", columns=[C.COUNTRY])
    val = load_fold("val", columns=[C.COUNTRY])
    fit_fold, tune_fold = inner_split(train)
    fit_sample = sample_s1(fit_fold.s1, cfg.n_fit_s1)[C.ENTITY_ID]
    mock = build_mock(train, val, tune_fold.s1[C.ENTITY_ID], fit_sample, target_shape())
    # the transliteration token map learned from train-fold pairs (cached; identical to the
    # map of v101, so this notebook needs no earlier version's artifacts)
    token_map = learn_token_map(cfg, train)
del val, fit_fold, tune_fold, fit_sample
absent = absent_from_mock(mock, train)
s1_ids = pd.Index(sample_s1(train.s1[isin(train.s1[C.ENTITY_ID], absent)], N_STAGE1)[C.ENTITY_ID])
print(f"absent from the mock: {len(absent):,}; stage-1 training entities: {len(s1_ids):,}")
train.s1[isin(train.s1[C.ENTITY_ID], s1_ids)][C.COUNTRY].value_counts()

## 4. Method

### 4.1 Stage 1 v2 on the GPU

Blocking of the training entities against the train-fold pool (cached), features chunk by
chunk to disk, then XGBoost on the GPU reading the chunks through a `DataIter`
(`stage1.fit_stage1`). The chunks are deleted once the model is saved.

In [ ]:
work = cfg.cache_dir / "stage1_chunks" / f"v106_{cfg.blocking.key()}"
t0 = time.time()
manifest = (json.loads((work / "manifest.json").read_text()) if (work / "manifest.json").exists()
            else write_chunks(cfg, train, s1_ids, token_map, work, timings=timings))
timings["stage1_set_seconds"] = round(time.time() - t0, 2)
print(f"training set {timings['stage1_set_seconds']:.0f} s: {manifest['rows']:,} pairs of "
      f"{manifest['entities']:,} entities, positive rate {manifest['positives'] / manifest['rows']:.3f}")
t0 = time.time()
s1_model = fit_stage1(manifest, S1_PARAMS)
timings["stage1_fit_seconds"] = round(time.time() - t0, 2)
stage1 = Fitted(s1_model, DecisionRule(), pd.DataFrame({"f_beta": [np.nan]}), cfg, token_map,
                {"stage1": "xgb gpu", "fit_info": s1_model.fit_info_})
stage1.save(ARTIFACTS / "stage1_v2")
clean(work)
print(json.dumps(s1_model.fit_info_, indent=1))
s1_model.importance().head(15).rename("gain share").to_frame()

### 4.2 Stage 1 v2 alone on the mock

The stage-1 pass keeps every pair with p1 ≥ 0.01 among its entity's 16 best: all pairs any
threshold rule could keep. The threshold rule on `p1`, tuned on the mock tune entities, gives
stage 1 v2's single-stage mock F0.5, comparable with v103 (v101 + mock rule).

In [ ]:
t0 = time.time()
outs = mock_stage1(cfg, stage1, mock, tcfg, timings=timings, cache_dir=STAGE1_CACHE / "mock")
timings["stage1_mock_seconds"] = round(time.time() - t0, 2)
from entity_resolution.decision import one_to_one_filter
keep_ids = pd.Index(mock.fold.s1[C.ENTITY_ID][mock.role.isin(["tune", "val"]).to_numpy()])
p1_scored = []
for o in outs.values():
    s = o.pairs.assign(prob=o.X["p1"].to_numpy())
    s = one_to_one_filter(s)
    p1_scored.append(s[isin(s[C.S1_ID], keep_ids)])
p1_scored = pd.concat(p1_scored, ignore_index=True)
rule1, _ = tune_mock(p1_scored, mock, cfg.grid, fp_weight=FP_WEIGHT)
single = mock_scores(p1_scored, mock, rule1)
del p1_scored
kept = pd.concat([o.pairs for o in outs.values()], ignore_index=True)
filter_report = pd.DataFrame({r: blocking_report(kept, mock.part(r)) for r in ("tune", "val")}).T
print(f"stage-1 pass {timings['stage1_mock_seconds']:.0f} s; rule {rule1}")
display(single[["f_beta", "f_tight", "est_public", "f_beta_singletons", "pair_precision",
                "pair_recall"]])
filter_report[["pair_recall", "entity_recall", "ceiling_f_beta", "candidates_mean", "candidates_p95"]]

### 4.3 Stage 2 on top, as in v104

Cross-fitted stage 2 on the mock's fit entities (GPU), stage-2 probabilities with the
pool-side 1-to-1 across every present entity, the rule tuned on the mock tune entities.

In [ ]:
t0 = time.time()
models, fit_info = fit_stage2(outs, mock, tcfg)
scored, report = mock_scored(outs, models, mock, tcfg)
scored.to_parquet(ARTIFACTS / "mock_scored.parquet", index=False)
rule_t, table_t = tune_mock(scored, mock, cfg.grid, fp_weight=FP_WEIGHT)
tune_part = mock.part("tune")
rows_t = scored[isin(scored[C.S1_ID], pd.Index(tune_part.s1[C.ENTITY_ID]))]
rule_e, table_e = tune_expected(rows_t, tune_part.s1[C.ENTITY_ID], tune_part.pairs,
                                gammas=(0.7, 0.85, 1.0, 1.2, 1.5, 2.0),
                                misses=(0.0, 0.05, 0.1, 0.2, 0.4), fp_weight=FP_WEIGHT)
rule, table = ((rule_e, table_e) if table_e["f_beta"].max() > table_t["f_beta"].max()
               else (rule_t, table_t))
timings["stage2_seconds"] = round(time.time() - t0, 2)
res = mock_scores(scored, mock, rule)
print(f"stage 2 {timings['stage2_seconds']:.0f} s; rule {rule} (tight-tuned)")
importance = pd.concat([m.importance() for m in models], axis=1).mean(axis=1)
display(importance.sort_values(ascending=False).head(15).rename("gain share").to_frame())
res[["f_beta", "f_tight", "est_public", "f_beta_singletons", "pair_precision", "pair_recall",
     "entities"]]

## 5. Evaluation

Mock F0.5 (val entities) of the single stage and the two-stage, against v103 and v104.

In [ ]:
cmp = pd.DataFrame({
    "v103 (public 0.961)": {"est_public": 0.961},
    "v107 (v104 + tight rule)": {"est_public": EST_TO_BEAT},
    "v106 stage 1 v2 single stage": {"est_public": single.loc["all", "est_public"],
                                     "mock_f05": single.loc["all", "f_beta"]},
    "v106 two-stage": {"est_public": res.loc["all", "est_public"],
                       "mock_f05": res.loc["all", "f_beta"]},
}).T
cmp["delta_vs_v107"] = cmp["est_public"] - EST_TO_BEAT
for c in ("India", "US"):
    cmp.loc["v106 two-stage", f"est_public_{c}"] = res.loc[c, "est_public"]
cmp.round(4)

### Ablation

Stage 2 retrained without the cohesion and rival columns (same parts, same early stopping,
tight-tuned rule): what those groups add on top of v104's competition and anchor features.

In [ ]:
from entity_resolution.stacking import COHESION_COLUMNS, RIVAL_COLUMNS
cols_all = list(next(iter(outs.values())).X.columns)
cols_base = [c for c in cols_all if c not in COHESION_COLUMNS and c not in RIVAL_COLUMNS]
t0 = time.time()
m_ab, _ = fit_stage2(outs, mock, tcfg, columns=cols_base)
s_ab, _ = mock_scored(outs, m_ab, mock, tcfg)
r_ab, _ = tune_mock(s_ab, mock, cfg.grid, fp_weight=FP_WEIGHT)
ab = mock_scores(s_ab, mock, r_ab).loc["all", ["f_beta", "est_public"]]
del m_ab, s_ab
ablation = pd.DataFrame({"full": res.loc["all", ["f_beta", "est_public"]],
                         "without cohesion/rivals": ab}).T
print(f"ablation {time.time() - t0:.0f} s")
ablation.round(4)

## 6. Error analysis

Error counts on the mock val entities for the two-stage model.

In [ ]:
part = mock.part("val")
matches = apply_rule(scored[isin(scored[C.S1_ID], pd.Index(part.s1[C.ENTITY_ID]))], rule)
counts = {k: len(error_samples(matches, part, k, n=10**9))
          for k in ("false_merge", "missed", "false_singleton", "singleton_merge")}
print("v106:", counts, "\nv104:", parent["metrics"].get("errors_mock"))

## 7. Log the result

In [ ]:
ts = TwoStage(stage1, models, rule, tcfg, table,
              {"fit": fit_info, "stage1_fit": s1_model.fit_info_,
               "filter_report": filter_report.to_dict("index")})
ts.save(ARTIFACTS)
record = {
    "hypothesis": "a GPU stage 1 trained on 3x the entities at train-fold density lifts the "
                  "two-stage model",
    "blocking_config": asdict(cfg.blocking), "stage1_params": asdict(S1_PARAMS),
    "stage1_cache": str(STAGE1_CACHE), "blocking_key": cfg.blocking.key(),
    "stage1_entities": len(s1_ids), "stage1_rows": manifest["rows"],
    "stage1_fit_info": s1_model.fit_info_, "single_stage_mock_f_beta": single.loc["all", "f_beta"],
    "single_stage_rule": asdict(rule1), "two_stage": tcfg.record(), "rule": asdict(rule),
    "rule_kind": type(rule).__name__, "fp_weight": FP_WEIGHT,
    "mock_f_beta": res.loc["all", "f_beta"], "est_public": res.loc["all", "est_public"],
    "f_tight": res.loc["all", "f_tight"], "single_stage_est_public": single.loc["all", "est_public"],
    **{f"mock_{c}": res.loc[c, "f_beta"] for c in ("India", "US")},
    **{k: res.loc["all", k] for k in ("f_beta_singletons", "f_beta_matched",
                                      "pair_precision", "pair_recall")},
    "cand_recall_val": filter_report.loc["val", "pair_recall"],
    "cands_mean_val": filter_report.loc["val", "candidates_mean"],
    "errors_mock": counts, "ablation": ablation.to_dict("index"), **timings,
    "peak_rss_gb": peak_rss_gb(),
}
DECISION = "KEEP" if record["est_public"] > EST_TO_BEAT + 0.001 else "DROP"
record["decision"] = DECISION
print(f"est_public v107 {EST_TO_BEAT:.4f} -> v106 {record['est_public']:.4f} {DECISION}")
row = log_result(
    EXP_DIR, change=f"two-stage with a GPU stage 1 (XGBoost, {N_STAGE1:,} entities absent from "
                    f"the mock), blocking {cfg.blocking.key()}",
    group="D4", mock_f05=record["mock_f_beta"], cand_recall=record["cand_recall_val"],
    notes=(f"est_public {record['est_public']:.4f}; single stage est "
           f"{record['single_stage_est_public']:.4f}; cands {record['cands_mean_val']:.1f}/S1; "
           f"blocking {cfg.blocking.key()}"),
    metrics=record, owner="M1", parent="v107", decision=DECISION)
row

## 8. Conclusion

Written after the run from the numbers above.

## 9. Test inference

`run_test_two_stage` with stage 1 v2 (stage-1 outputs cached per stage 1, blocking and
filter), both validators, files kept in `submissions/v106/`.

In [ ]:
t0 = time.time()
match_path, cand_path, s1n_test, test_matches, test_summary = run_test_two_stage(
    cfg, ts, cache_dir=STAGE1_CACHE / "test")
print(f"run_test {time.time() - t0:.0f} s")
country_of = s1n_test.set_index(C.ENTITY_ID)[C.COUNTRY]
n_s1 = s1n_test.groupby(C.COUNTRY).size()
by = test_matches[C.S1_ID].map(country_of)
display(pd.DataFrame({
    "s1": n_s1,
    "cands_per_s1": test_summary["n_cands"].groupby(test_summary.index.map(country_of)).sum() / n_s1,
    "matched_share": test_matches.groupby(by)[C.S1_ID].nunique() / n_s1,
    "matches_per_s1": test_matches.groupby(by).size() / n_s1}))
dest = C.ROOT / "submissions" / "v106"
dest.mkdir(parents=True, exist_ok=True)
for p in (match_path, cand_path):
    shutil.copy2(p, dest / p.name)
out = subprocess.run([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                      str(C.OUTPUT), "--check-ids"], capture_output=True, text=True)
print(out.stdout[-2000:], out.stderr[-2000:])
out = subprocess.run([sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                      "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")],
                     capture_output=True, text=True)
print(out.stdout[-3000:], out.stderr[-2000:])
print(f"notebook total {time.time() - t_start:.0f} s, peak RSS {peak_rss_gb()} GB")